In [ ]:
!pip install pandas openpyxl scikit-learn matplotlib seaborn numpy

In [ ]:
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# ====================== 配置项 ======================
DATA_PATH = r"C:\Users\33759\Desktop\陈正扬\处理后数据（不含学科）\合并数据（不含学科）.xlsx"

SCATTER_SAVE_PATH = r"项目聚类散点图.png"
RADAR_SAVE_PATH = r"项目聚类雷达图.png"

CLUSTER_RANGE = range(6, 16)
MAX_FEATURES = 5000
MIN_DF = 3
MAX_DF = 0.85
RADAR_NUM_FEATURES = 12

# ====================== 字体设置 ======================
# 只使用 Windows 自带的中文字体
plt.rcParams["font.family"] = ["SimHei", "Microsoft YaHei", "SimSun"]
plt.rcParams["axes.unicode_minus"] = False

# ====================== 停用词表（与文本挖掘保持一致） ======================
stopwords = {
    "研究", "基于", "的", "与", "和", "对", "及", "等", "从", "论", "一种", "路径", "机制",
    "模式", "体系", "构建", "实践", "应用", "探究", "考察", "梳理", "阐释", "解读", "思考",
    "初探", "及其", "关于", "以", "为", "浅谈", "试论", "探讨", "探析", "刍议", "略论"
}

# ====================== 1. 数据读取 ======================
df = pd.read_excel(DATA_PATH)
print(f"原始项目数: {len(df)}")

# ====================== 2. 文本预处理 ======================
def preprocess_text(text):
    if pd.isna(text):
        return ""
    # 只保留中文，标点数字英文全替换为空格
    text = re.sub(r"[^\u4e00-\u9fa5]", " ", str(text))
    text = re.sub(r"\s+", " ", text).strip()

    # 使用 jieba 分词
    import jieba
    words = jieba.lcut(text)

    # 过滤停用词 + 长度小于2的词
    words = [w for w in words if w not in stopwords and len(w) >= 2]
    return " ".join(words)

df["clean_title"] = df["项目名称"].apply(preprocess_text)

# 检查清洗效果
empty_count = (df["clean_title"] == "").sum()
print(f"清洗后空标题数量: {empty_count}")

all_text = " ".join(df["clean_title"])
word_counts = Counter(all_text.split())
print("Top 30 高频主题词（确认清洗正常）:", word_counts.most_common(30))

df = df[df["clean_title"] != ""].reset_index(drop=True)
print(f"最终有效项目数: {len(df)}")

# ====================== 3. TF-IDF 向量化 ======================
vectorizer = TfidfVectorizer(max_features=MAX_FEATURES, min_df=MIN_DF, max_df=MAX_DF)
tfidf_matrix = vectorizer.fit_transform(df["clean_title"])
features = vectorizer.get_feature_names_out()
print(f"TF-IDF 词汇量: {len(features)}")

# ====================== 4. 寻找最佳聚类数 ======================
print("\n正在计算轮廓系数寻找最佳K...")
sil_scores = []
for k in CLUSTER_RANGE:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = kmeans.fit_predict(tfidf_matrix)
    score = silhouette_score(tfidf_matrix, labels)
    sil_scores.append((k, score))
    print(f"K={k}, 轮廓系数: {score:.4f}")

best_k = max(sil_scores, key=lambda x: x[1])[0]
print(f"\n最佳聚类数: {best_k}")

# ====================== 5. 最终聚类 ======================
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init="auto")
df["cluster"] = kmeans.fit_predict(tfidf_matrix)

# ====================== 6. PCA 散点图 ======================
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(tfidf_matrix.toarray())
df["pca_x"] = pca_coords[:, 0]
df["pca_y"] = pca_coords[:, 1]

plt.figure(figsize=(12, 9))
sns.scatterplot(data=df, x="pca_x", y="pca_y", hue="cluster", palette="deep",
                alpha=0.7, s=60, edgecolor="none")
plt.title(f"国家社科立项项目标题聚类散点图 (K={best_k}, 解释方差 {pca.explained_variance_ratio_.sum():.1%})", fontsize=16)
plt.xlabel("PCA 主成分1")
plt.ylabel("PCA 主成分2")
plt.legend(title="聚类簇", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

if SCATTER_SAVE_PATH:
    plt.savefig(SCATTER_SAVE_PATH, dpi=300, bbox_inches="tight")
    print(f"\n散点图已保存: {SCATTER_SAVE_PATH}")
plt.show()

# ====================== 7. 各簇深度解读 ======================
print("\n" + "="*80)
print("各聚类簇深度解读")
print("="*80)

for i in range(best_k):
    cluster_df = df[df["cluster"] == i]
    size = len(cluster_df)
    pct = size / len(df) * 100

    print(f"\n【簇 {i}】 规模: {size} 个项目 ({pct:.1f}%)")
    print("   年份分布:", dict(cluster_df["立项年份"].value_counts().sort_index()))
    print("   项目类别Top3:", dict(cluster_df["项目类别"].value_counts().head(3)))

    # 簇内Top15专属关键词
    cluster_tfidf = tfidf_matrix[cluster_df.index].mean(axis=0).A1
    top_idx = cluster_tfidf.argsort()[-15:][::-1]
    top_words = [(features[j], round(cluster_tfidf[j], 4)) for j in top_idx]
    print("   专属核心关键词Top15:")
    for w, s in top_words:
        print(f"     - {w} ({s})")



In [ ]:
# ====================== 8. 雷达图 ======================
print("\n正在准备雷达图数据...")

# 候选高频词（取前20个备用）
candidate_words = [w for w, _ in word_counts.most_common(20)]

# 收集有效维度
valid_words = []
valid_indices = []
feature_set = set(features)

for word in candidate_words:
    if word in feature_set:
        idx = np.where(features == word)[0][0]
        valid_words.append(word)
        valid_indices.append(idx)
    if len(valid_words) >= 12:
        break

# 如果不够，从全局高频特征补齐
if len(valid_words) < 8:
    print("高频词不足，自动补充...")
    overall_mean = np.ravel(tfidf_matrix.mean(axis=0))
    top_idxs = overall_mean.argsort()[-20:][::-1]
    for idx in top_idxs:
        word = features[idx]
        if word not in valid_words:
            valid_words.append(word)
            valid_indices.append(idx)
        if len(valid_words) >= 12:
            break

print(f"雷达图使用 {len(valid_words)} 个核心主题词: {valid_words}")

# ====================== 强制转密集矩阵，避免稀疏坑 ======================
# 先把整个TF-IDF矩阵转成密集numpy array
tfidf_dense = tfidf_matrix.toarray()  # 解决稀疏索引问题

# 计算每个簇在选定维度上的平均值
cluster_means = []
for i in range(best_k):
    cluster_mask = df["cluster"] == i
    cluster_data = tfidf_dense[cluster_mask]
    if len(cluster_data) == 0:
        mean_vals = np.zeros(len(valid_indices))
    else:
        mean_vals = cluster_data.mean(axis=0)[valid_indices]  # 直接索引
    cluster_means.append(mean_vals)

cluster_means = np.array(cluster_means)

# 标准化：每簇最大值为1
row_max = cluster_means.max(axis=1, keepdims=True)
cluster_norm = np.where(row_max > 0, cluster_means / row_max, 0)

# ====================== 绘图 ======================
angles = np.linspace(0, 2 * np.pi, len(valid_words), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(12, 12), subplot_kw=dict(polar=True))

# 颜色参数修改
colors = plt.cm.Set3(np.linspace(0, 1, best_k))

for i in range(best_k):
    values = np.append(cluster_norm[i], cluster_norm[i][0])
    ax.plot(angles, values, 'o-', linewidth=3, label=f"簇 {i} ({len(df[df['cluster']==i])}项)", color=colors[i])
    ax.fill(angles, values, alpha=0.2, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(valid_words, fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=10, color="grey")
ax.grid(True, linestyle='--', alpha=0.8)
ax.set_title("国家社科立项项目研究主题聚类雷达图\n（各簇在核心热点词上的强度对比）",
             fontsize=20, fontweight='bold', pad=50)

ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.0), fontsize=12, frameon=True, fancybox=True)

if RADAR_SAVE_PATH:
    plt.savefig(RADAR_SAVE_PATH, dpi=400, bbox_inches="tight", facecolor='white')
    print(f"\n雷达图已保存至: {RADAR_SAVE_PATH}")

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("="*60)

In [ ]:
# ====================== 热点主题随时间演变堆叠面积图 ======================
print("\n正在生成热点主题随时间演变堆叠面积图...")

import matplotlib.pyplot as plt
import seaborn as sns

# 按年份和簇统计占比（行归一化）
pivot_table = pd.crosstab(df["立项年份"], df["cluster"], normalize='index') * 100
pivot_table = pivot_table.sort_index()  # 确保年份顺序

# 绘图
plt.figure(figsize=(12, 7))
pivot_table.plot(kind='area', stacked=True, alpha=0.8, cmap='tab20', linewidth=0.5)

plt.title("2020-2025年各主题簇项目占比堆叠面积图", fontsize=16, fontweight='bold')
plt.xlabel("立项年份", fontsize=14)
plt.ylabel("各主题簇占比 (%)", fontsize=14)
plt.legend(title="主题簇号", bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10, ncol=1)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()

# 保存高清图
TIME_EVOLUTION_SAVE_PATH = r"热点演变堆叠面积图.png"  # 可修改路径
plt.savefig(TIME_EVOLUTION_SAVE_PATH, dpi=400, bbox_inches='tight', facecolor='white')
print(f"热点演变堆叠面积图已保存至: {TIME_EVOLUTION_SAVE_PATH}")
plt.show()